In [ ]:
import pandas as pd
import geopandas as gp
import numpy as np
import matplotlib.pyplot as plt
import scipy as st
import seaborn as sns

In [ ]:
df = pd.read_csv("/Users/aangphurbasherpa/Desktop/NYC Taxi Duration/data/processed/NYC_ride_cleaned.csv")
pd.set_option('display.float_format', '{:.2f}'.format)

In [ ]:
df.head(5)

Checking for any Nulls in the data

In [ ]:
df.isna().sum()

In [ ]:
df.describe()

In [ ]:

geometry  = gp.points_from_xy(df['dropoff_longitude'],df['dropoff_latitude'])
from scipy.ndimage import gaussian_filter
gdf  = gp.GeoDataFrame(df,geometry=geometry,crs='EPSG:4326')
x = gdf['dropoff_longitude'].values
y = gdf['dropoff_latitude'].values
xy = np.vstack([x,y])
density, xedges, yedges = np.histogram2d(x, y, bins=600)
density_smooth = gaussian_filter(density, sigma=0.4)
plt.figure(figsize=(12,7))
fig = plt.gcf()
ax = plt.gca()
fig.patch.set_facecolor('black')
ax.set_facecolor('black')
plt.imshow(
    np.rot90(density_smooth),
    cmap ='inferno',
    extent=[xedges[0], xedges[-1], yedges[0], yedges[-1]],
    aspect = 'auto',
    norm = plt.matplotlib.colors.LogNorm(vmin=1)
)

From the above plot we can see where the majority of the taxi rides takes place in the city

Now using haversine distance formula to calculate distance between the pickup and dropoff points as a feature

$$
\begin{aligned}
a &= \sin^2\left(\frac{\Delta \varphi}{2}\right) + \cos(\varphi_1) \cos(\varphi_2) \sin^2\left(\frac{\Delta \lambda}{2}\right) \\
c &= 2 \arcsin\left(\sqrt{a}\right) \\
d &= R \cdot c
\end{aligned}
$$

The formula in Python

In [ ]:
def haversine_form(lat1,long1,lat2,long2):
    lat1 , long1,lat2,long2 = map(np.radians, [lat1, long1, lat2, long2])
    dlat = lat2-lat1
    dlong = long2-long1
    a  = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2)*np.sin(dlong/2)**2
    c = 2 * np.arcsin(np.sqrt(a))
    r = 3956
    return c*r

In [ ]:
df['distance'] = haversine_form(df['pickup_latitude'],df['pickup_longitude'],df['dropoff_latitude'],df['dropoff_longitude'])

Now plotting the distance and trip_duration to look for any patterns

In [ ]:
print(df['trip_duration'].describe())
print(df['distance'].describe())

The distance in miles has a minimum distance of 0.0 miles which is a bad recording of the data
these outliers are removed 

In [ ]:
df = df[df['distance'] > 0]            
df = df[df['distance'] <= 30]          

In [ ]:
fig, axes = plt.subplots(1,2,figsize=(12,7))
axes[0].hist(df['distance'],bins=20)
axes[0].set_title("Distance")
axes[1].hist(df['trip_duration'],bins=20)
axes[1].set_title("Duration of trip")

In [ ]:
feature = ['trip_duration','distance']
df[feature].corr(method='pearson')

In [ ]:
df['speed'] = df['distance'] /df['trip_duration']